# The three lead `@wasm` use cases (v0.2.5 M1.5)

1. **Non-vectorisable ML preprocessing / decoding** — edit distance and Viterbi decoding, pure-Python DP that NumPy cannot express, run in-process on the server by the same `@wasm` decorator (the M1 image-preprocess demo is the browser instance of this row).
2. **Sandboxed (LLM-generated) code** — a generated snippet compiled and run under fuel + no I/O; the escape attempts are shown being contained.
3. **Isomorphic: same function, browser and server** — `dtw_distance` runs in a real tab and in-process; the bits are identical.

Every number is computed by `features_lib.py`; see `full_features.ipynb` for every row incl. "where it loses". Setup: `pip install -e .[test]`, `python -m pythscribe.build examples/wasm-use-cases/kernels.py`.

In [1]:
import os, sys, json, time
from pathlib import Path
from IPython.display import Markdown, display
HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import features_lib as F

FAST = os.environ.get("NB_FAST", "0") == "1"      # the test runner shrinks the sizes; the numbers are still measured
K = F.load_kernels(HERE / "kernels.py")
modes = {name: (F.binding(getattr(K, name)).mode, F.binding(getattr(K, name)).mode_reason)
         for name in ("edit_distance", "dtw_distance", "viterbi", "mask_digit_runs", "spin", "sum_squares", "is_vowel", "count_vowels")}
for name, (mode, why) in modes.items():
    print(f"{name:16s} mode={mode:8s} {why}")
assert all(m == "server" for m, _ in modes.values()), "build the artifacts first: python -m pythscribe.build examples/wasm-use-cases/kernels.py (and pip install wasmtime)"
records = {}

edit_distance    mode=server   auto: artifact + wasmtime + FFI grammar
dtw_distance     mode=server   auto: artifact + wasmtime + FFI grammar
viterbi          mode=server   auto: artifact + wasmtime + FFI grammar
mask_digit_runs  mode=server   auto: artifact + wasmtime + FFI grammar
spin             mode=server   auto: artifact + wasmtime + FFI grammar
sum_squares      mode=server   auto: artifact + wasmtime + FFI grammar
is_vowel         mode=server   auto: artifact + wasmtime + FFI grammar
count_vowels     mode=server   auto: artifact + wasmtime + FFI grammar


## 1. Non-vectorisable decoding on the server

In [2]:
records["speedup"] = F.measure_speedup(K, n=200 if FAST else 800, seed=0, repeats=3)
r = records["speedup"]
print(f"edit_distance {r['n']}x{r['n']}: {r['ratio']:.1f}x vs interpreted CPython, all paths agree: {r['all_equal']}")
# Viterbi: the decoder, with its state path written into a caller-provided list (in place, Python semantics)
# a decoder-sized problem (12 states x 1500 steps): the DP work must dominate the one-time marshalling of the score/backpointer buffers
inp = F._viterbi_inputs(12, 300 if FAST else 1500, 0)
t0 = time.perf_counter(); best_w, path_w = F.viterbi_call(K.viterbi, inp); tw = time.perf_counter() - t0
t0 = time.perf_counter(); best_p, path_p = F.viterbi_call(F.binding(K.viterbi).run_python, inp); tp = time.perf_counter() - t0
records["viterbi"] = {"n_states": 12, "n_steps": inp["n_steps"], "wasm_s": tw, "cpython_s": tp, "ratio": tp / tw, "same_path": path_w == path_p, "same_score_bits": F.digest(best_w, path_w) == F.digest(best_p, path_p)}
print("viterbi:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in records["viterbi"].items()})
assert records["viterbi"]["same_path"] and records["viterbi"]["same_score_bits"]

edit_distance 800x800: 13.1x vs interpreted CPython, all paths agree: True
viterbi: {'n_states': 12, 'n_steps': 1500, 'wasm_s': 0.0144, 'cpython_s': 0.0284, 'ratio': 1.9776, 'same_path': True, 'same_score_bits': True}


## 2. Sandboxed (LLM-generated) code — and the escape attempts (this cell FAILS if one escapes)

In [3]:
print(F.LLM_SNIPPET)
records["sandbox"] = F.measure_sandbox(K, fuel=20_000_000, workdir=HERE / "_sandbox_work")
sb = records["sandbox"]
print("snippet ran under fuel:", sb["snippet"])
print("unbounded loop:", "TRAPPED" if sb["fuel_trap"]["trapped"] else "ESCAPED", f"in {sb['fuel_trap']['elapsed_s']*1e3:.0f} ms;", "RED half:", sb["fuel_trap"]["red_control"]["outcome"])
print("WASI file I/O import:", "REFUSED" if sb["io_attempt"]["refused"] else "ESCAPED", "; RED half: WASI linker wrote the file =", sb["io_attempt"]["red_control"]["wrote_file_under_wasi_linker"])
print("open() in a kernel:", "REFUSED at build" if sb["open_in_kernel"]["refused_at_build"] else "BUILT?!", "-", sb["open_in_kernel"]["error"][:120])
assert sb["contained"], "AN ESCAPE SUCCEEDED"
print("contained:", sb["contained"])

from pythscribe import wasm


@wasm
def score_tokens(ids: list[int], weights: list[int], k: int) -> int:
    # (representative LLM-generated scoring snippet) weighted count of ids below k, position-decayed
    total = 0
    n = len(ids)
    for i in range(n):
        if ids[i] < k:
            w = weights[ids[i] % len(weights)]
            total = total + w * (n - i)
    return total % 1000003



snippet ran under fuel: {'value': 342583, 'cpython_value': 342583, 'equal': True, 'fuel_used': 2250075, 'host_imports': [], 'wasm_bytes': 766}
unbounded loop: TRAPPED in 2 ms; RED half: interrupted by epoch after 0.30s (INTERRUPT)
WASI file I/O import: REFUSED ; RED half: WASI linker wrote the file = True
open() in a kernel: REFUSED at build - pyths compile failed for `leak`:
contained: True


## 3. Isomorphic — the same `@wasm` function in the tab and in-process, bit for bit

The real-tab run needs `gradio` + `playwright` (recorded NOT RUN otherwise); the engine-level identity below (browser shim under V8, wasmtime, CPython) needs only Node.

In [4]:
try:
    import iso_drive as drive
    records["isomorphic"] = drive.measure_isomorphic([0.0, 1.5, 2.25, 3.0, 2.5, 1.0, -0.5, 0.1], [0.0, 0.2, 1.0, 2.0, 3.1, 3.0, 2.0, 1.0, 0.0, -0.1], app_dir=HERE)
    iso = records["isomorphic"]
    print("tab   :", iso["browser_bits"], iso["path"]); print("server:", iso["server_bits"]); print("python:", iso["cpython_bits"]); print("identical:", iso["identical"], "| server-side work for the tab's answer: python_calls =", iso["python_calls"], "server_calls =", iso["server_calls"])
    assert iso["identical"] and iso["python_calls"] == 0 and iso["server_calls"] == 0
except ImportError as e:
    records["isomorphic"] = None; print("real-tab run NOT RUN:", e)
records["determinism"] = F.measure_determinism(K, runs=3, n_states=6, n_steps=100 if FAST else 400, seed=0)
de = records["determinism"]
print("viterbi digest identical across runs:", de["identical_across_runs"], "| == CPython:", de["equal_cpython"], "| == Node/V8 shim:", de["equal_node_v8"])
summary = {"speedup_ratio": r["ratio"], "viterbi_ratio": records["viterbi"]["ratio"], "sandbox_contained": sb["contained"],
           "isomorphic_identical": (records["isomorphic"] or {}).get("identical"), "determinism_three_engines": bool(de["identical_across_runs"] and de["equal_cpython"] and de["equal_node_v8"] is not False)}
(HERE / "three_use_cases_summary.json").write_text(json.dumps({"records": records, "summary": summary}, indent=2, sort_keys=True) + "\n", encoding="utf-8", newline="\n")
print(json.dumps(summary, indent=2)); print("computed, not constants")

tab   : 0000000000000240 browser-wasm
server: 0000000000000240
python: 0000000000000240
identical: True | server-side work for the tab's answer: python_calls = 0 server_calls = 0
viterbi digest identical across runs: True | == CPython: True | == Node/V8 shim: True
{
  "speedup_ratio": 13.107553793096827,
  "viterbi_ratio": 1.9775995239562747,
  "sandbox_contained": true,
  "isomorphic_identical": true,
  "determinism_three_engines": true
}
computed, not constants
